# 02 — Cleaning the per-minute sensor data

Code: `src/clean.py`. Steps (in order):

| # | Problem | Fix | Why |
|---|---|---|---|
| 1 | `-999.25` values | → NaN | Standard oilfield null code (LAS format). Averaging it in would ruin daily means. |
| 2 | Exact duplicate rows | drop | Same minute recorded twice — would double-weight that minute. |
| 3 | Spikes (1–3 min) | rolling median filter → NaN | Rock/fluid pressure can't jump 5000 psi for one minute: sensor glitch. |
| 4 | Choke only stored as change events | as-of join to every minute, `shut_in = choke == 0` | Needed for features (VFM) and to find build-ups (PTA). |

**Rolling median filter:**  `dev_t = |x_t − median(x_{t−5..t+5})|`, spike if `dev_t > 20 × typical dev` of that well.
The median ignores short spikes but keeps real steps and smooth build-up curves.

Checked: no real *frozen* sensors — the only long flat runs are gas lift = 0 during shut-ins (real).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('..', 'src').resolve()))
import clean
import load

hf = load.load_high_freq()
choke = load.load_choke_events()

df, report = clean.clean_high_freq(hf, choke)
df.to_parquet(load.PROCESSED / 'high_freq_clean.parquet', index=False)

for step, result in report.items():
    print(f'{step:20s} {result}')

## Check: shut-in minutes
The 6 shut-ins in the choke events (choke = 0 %) last 72 + 5.65 + 48 + 8 + 72 + 9 = 214.65 h = 12,879 min;
the cleaned minute data must flag exactly this many shut-in minutes.

In [ ]:
shut = df[df.shut_in].groupby('well_name').timestamp.agg(['min', 'max', 'count'])
shut['hours'] = shut['count'] / 60
print(shut)
print('total minutes:', shut['count'].sum())

## Before (grey) vs after (blue)

In [ ]:
def plot_before_after(well, start=None, end=None):
    raw = hf[hf.well_name == well].set_index('timestamp').loc[start:end]
    new = df[df.well_name == well].set_index('timestamp').loc[start:end]
    fig, axes = plt.subplots(len(clean.SENSORS), 1, figsize=(15, 14), sharex=True)
    for ax, col in zip(axes, clean.SENSORS):
        ax.plot(raw.index, raw[col].mask(raw[col] < -900), lw=0.4, color='lightgray', label='raw')
        ax.plot(new.index, new[col], lw=0.4, label='clean')
        ax.set_ylabel(col, fontsize=8)
    axes[0].legend(loc='upper right')
    fig.suptitle(f'{well}: raw vs clean')
    fig.tight_layout()
    plt.show()

plot_before_after('GL-01')

In [ ]:
# Zoom: a temperature spike removed right before a choke change (the choke step itself is kept)
plot_before_after('GL-06', '2025-09-19 00:00', '2025-09-19 04:00')